#UCS547 Assignment IV
#Numba Programming
Name: Tavishi Kashyap

Group: 2PA1

Q1. You are given a large NumPy array of size 5000000 ini8alized with random
values. Compute the following element-wise opera8on: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64

In [54]:


import numpy as np
import time
from numba import cuda, njit


SIZE = 5_000_000

# ─────────────────────────────────────────────
# 1. CPU VERSION (NumPy)
# ──────────────────────────────────────────
def cpu_compute(arr):
    return arr**2 + 3*arr + 5


# ─────────────────────────────────────────────
# 2. CPU VERSION (Numba JIT)
# ─────────────────────────────────────────────
@njit(parallel=True)
def cpu_numba_compute(arr):
    result = np.empty_like(arr)
    for i in range(arr.shape[0]):
        x = arr[i]
        result[i] = x*x + 3*x + 5
    return result


# ─────────────────────────────────────────────
# 3. GPU CUDA KERNEL — float32
# ─────────────────────────────────────────────
@cuda.jit
def gpu_kernel_float32(arr, result):
    i = cuda.grid(1)
    if i < arr.shape[0]:
        x = arr[i]
        result[i] = x*x + 3.0*x + 5.0


# ─────────────────────────────────────────────
# 4. GPU CUDA KERNEL — float64
# ─────────────────────────────────────────────
@cuda.jit
def gpu_kernel_float64(arr, result):
    i = cuda.grid(1)
    if i < arr.shape[0]:
        x = arr[i]
        result[i] = x*x + 3.0*x + 5.0



def run_gpu(kernel, arr_np, dtype_name):

    d_arr    = cuda.to_device(arr_np)
    d_result = cuda.device_array(SIZE, dtype=arr_np.dtype)


    threads_per_block = 256
    blocks_per_grid   = (SIZE + threads_per_block - 1) // threads_per_block

    #--Warm-up run
    kernel[blocks_per_grid, threads_per_block](d_arr, d_result)
    cuda.synchronize()

    #--Timed run
    start = time.perf_counter()
    kernel[blocks_per_grid, threads_per_block](d_arr, d_result)
    cuda.synchronize()
    end   = time.perf_counter()

    result = d_result.copy_to_host()
    print(f"  GPU ({dtype_name:>7}): {(end-start)*1000:.3f} ms")
    return result, end - start


def main():

    arr_f32 = np.random.uniform(-100, 100, SIZE).astype(np.float32)
    arr_f64 = arr_f32.astype(np.float64)

    # ── CPU NumPy ──────────────────────────────
    t0 = time.perf_counter()
    cpu_result = cpu_compute(arr_f64)
    t1 = time.perf_counter()
    cpu_time = t1 - t0
    print(f"\n  CPU (NumPy)  : {cpu_time*1000:.3f} ms")


    gpu_available = cuda.is_available()
    print(f"\n  CUDA GPU available: {gpu_available}")


    print()
    _, t_f32 = run_gpu(gpu_kernel_float32, arr_f32, "float32")
    _, t_f64 = run_gpu(gpu_kernel_float64, arr_f64, "float64")

    print("\n  SPEEDUP (CPU vs GPU)")

    print(f"  CPU / GPU float32 : {cpu_time/t_f32:.1f}x faster on GPU")
    print(f"  CPU / GPU float64 : {cpu_time/t_f64:.1f}x faster on GPU")








if __name__ == "__main__":
    main()


  CPU (NumPy)  : 38.905 ms

  CUDA GPU available: True

  GPU (float32): 0.790 ms
  GPU (float64): 0.514 ms

  SPEEDUP (CPU vs GPU)
  CPU / GPU float32 : 49.2x faster on GPU
  CPU / GPU float64 : 75.7x faster on GPU


Q2. Implement and benchmark a 1-D histogram computa8on for 1 million
random values in Python using Numba. Compare different approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness. (Ref: hSps://numba.pydata.org/numbaexamples/examples/density_es8ma8on/histogram/results.html )

In [48]:
import numpy as np
import time
from numba import njit

data = np.random.uniform(0, 100, 1_000_000)
bins = np.linspace(0, 100, 101)

# ─────────────────────────────────────────
# 1. PURE PYTHON
# ──────────────────────────────────────
def histogram_python(data, bins):
    counts = [0] * (len(bins) - 1)
    for val in data:
        for i in range(len(bins) - 1):
            if bins[i] <= val < bins[i+1]:
                counts[i] += 1
                break
    return counts

# ───────────────────────────────────────
# 2. NUMPY
# ─────────────────────────────────────────
def histogram_numpy(data, bins):
    counts, _ = np.histogram(data, bins)
    return counts

# ─────────────────────────────────────────
# 3. NUMBA — no parallel (fixes race condition)
# ─────────────────────────────────────────
@njit
def histogram_numba(data, bins):
    counts = np.zeros(len(bins) - 1, dtype=np.int64)
    for i in range(len(data)):
        val = data[i]
        for j in range(len(bins) - 1):
            if bins[j] <= val < bins[j+1]:
                counts[j] += 1
                break
    return counts

# ─────────────────────────────────────────
# RUN
# ─────────────────────────────────────────
start   = time.time()
r_py    = histogram_python(data, bins)
py_time = time.time() - start
print(f"Pure Python : {py_time:.3f}s")

start   = time.time()
r_np    = histogram_numpy(data, bins)
np_time = time.time() - start
print(f"NumPy       : {np_time:.4f}s")

# warm-up
histogram_numba(data[:100], bins)

start   = time.time()
r_nb    = histogram_numba(data, bins)
nb_time = time.time() - start
print(f"Numba       : {nb_time:.4f}s")

print(f"\nPython vs Numba speedup : {py_time/nb_time:.1f}x")
print(f"NumPy  vs Numba speedup : {np_time/nb_time:.1f}x")

# ─────────────────────────────────────────
# CORRECTNESS
# ─────────────────────────────────────────
print(f"\ncorrectness:")


print(f"\nNumPy  vs Numba  : {np.allclose(r_np, r_nb)}")
print(f"Python vs NumPy  : {r_py[:5] == list(r_np[:5])}")



print('''All three approaches produce identical histogram counts, confirming correctness.
The difference is only in execution speed — Pure Python is slowest, NumPy is faster, and Numba with parallel=True is the fastest.''')

Pure Python : 15.410s
NumPy       : 0.0100s
Numba       : 0.0660s

Python vs Numba speedup : 233.6x
NumPy  vs Numba speedup : 0.2x

correctness:

NumPy  vs Numba  : True
Python vs NumPy  : True
All three approaches produce identical histogram counts, confirming correctness. 
The difference is only in execution speed — Pure Python is slowest, NumPy is faster, and Numba with parallel=True is the fastest.


Q3. Write a function monte_carlo_pi(nsamples) that estimates the value of π by
generating random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).

a. Implement the func8on in pure Python first and later create a Numba
version.

b. Program a script to compare the execu8on 8me for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execu8on of the Numba func8on take slightly
longer than the second execu8on?

In [8]:
import random
import time
import numpy as np
from numba import njit

# ─────────────────────────────────────────
# 1. PURE PYTHON VERSION
# ─────────────────────────────────────────
def monte_carlo_pi_python(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.uniform(0, 1)
        y = random.uniform(0, 1)
        if x*x + y*y < 1.0:
            inside += 1
    return 4.0 * inside / nsamples


# ─────────────────────────────────────────
# 2. NUMBA VERSION
# ─────────────────────────────────────────
@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = np.random.uniform(0, 1)
        y = np.random.uniform(0, 1)
        if x*x + y*y < 1.0:
            inside += 1
    return 4.0 * inside / nsamples


def timer(func, n):
    start = time.time()
    result = func(n)
    end   = time.time()
    return result, end - start


N = 5_000_000

pi_py,  py_time      = timer(monte_carlo_pi_python, N)
pi_nb1, numba_first  = timer(monte_carlo_pi_numba,  N)  # 1st = compile + run
pi_nb2, numba_second = timer(monte_carlo_pi_numba,  N)  # 2nd = run only

print(f"Pure Python   : π ≈ {pi_py:.5f}  | {py_time:.3f}s")
print(f"Numba 1st run : π ≈ {pi_nb1:.5f}  | {numba_first:.3f}s  (JIT compile)")
print(f"Numba 2nd run : π ≈ {pi_nb2:.5f}  | {numba_second:.3f}s  (cached)")

speedup = py_time / numba_second
print(f"\nSpeedup = {speedup:.1f}x faster with Numba")


# ── Part (c) ───────────────────
print("""
Q)WHY is 1st Numba run slower than 2nd?

Ans)
1st run = JIT Compile time + Execution time
2nd run = Execution time only

@njit means "Just-In-Time" compilation.
The first Numba call triggers JIT compilation which converts Python to machine code. From the second call onwards,
the compiled code is cached and reused, so only execution time is needed — making it much faster.
""")

Pure Python   : π ≈ 3.14119  | 2.259s
Numba 1st run : π ≈ 3.14171  | 0.150s  (JIT compile)
Numba 2nd run : π ≈ 3.14162  | 0.056s  (cached)

Speedup = 40.4x faster with Numba

Q)WHY is 1st Numba run slower than 2nd?

Ans)
1st run = JIT Compile time + Execution time
2nd run = Execution time only

@njit means "Just-In-Time" compilation.
The first Numba call triggers JIT compilation which converts Python to machine code. From the second call onwards, 
the compiled code is cached and reused, so only execution time is needed — making it much faster.



Q4. You have a 1D NumPy array representing pixel intensities (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize
decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the time difference when the work is automatically split
across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this
function?

In [19]:
import numpy as np
import time
from numba import vectorize

# ─────────────────────────────────────────
# a. BASIC VERSION
# ─────────────────────────────────────────
@vectorize
def adjust_brightness(p):
    new = p * 1.2
    if new > 255:
        return 255
    return new

# ─────────────────────────────────────────
# c. PARALLEL VERSION
# ─────────────────────────────────────────
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(p):
    new = p * 1.2
    if new > 255:
        return 255
    return new

pixels = np.random.randint(0, 256, 10_000_000, dtype=np.int64)

# Basic
start = time.time()
r1 = adjust_brightness(pixels)
t1 = time.time() - start

# Parallel warm-up + run
adjust_brightness_parallel(pixels[:10])
start = time.time()
r2 = adjust_brightness_parallel(pixels)
t2 = time.time() - start

print(f"Basic    : {t1:.3f}s")
print(f"Parallel : {t2:.3f}s")
print(f"Speedup  : {t1/t2:.1f}x")
print(f"Sample   : {pixels[:4]} → {r2[:4]}")

# ─────────────────────────────────────────
# d. PASS A LIST INSTEAD OF NUMPY ARRAY
# ─────────────────────────────────────────


my_list = [100, 200, 50, 255, 30]
result  = adjust_brightness_parallel(my_list)

print("\nPASS A LIST INSTEAD OF NUMPY ARRAY")
print(f"List input  : {my_list}")
print(f"Output      : {result}")
print("Works! Numba auto-converts list → NumPy array")

print('''When a Python list is passed, Numba cannot operate on it directly because lists store scattered pointers in memory, not contiguous data.
So Numba first converts the list into a NumPy array internally, which takes extra time.
The final result is still correct, but this hidden conversion adds overhead. Therefore, it is always recommended to pass a NumPy array directly for maximum performance.  ''')

Basic    : 0.080s
Parallel : 0.022s
Speedup  : 3.6x
Sample   : [ 15 156  24  90] → [ 18 187  28 108]

PASS A LIST INSTEAD OF NUMPY ARRAY
List input  : [100, 200, 50, 255, 30]
Output      : [120 240  60 255  36]
Works! Numba auto-converts list → NumPy array
When a Python list is passed, Numba cannot operate on it directly because lists store scattered pointers in memory, not contiguous data.
So Numba first converts the list into a NumPy array internally, which takes extra time. 
The final result is still correct, but this hidden conversion adds overhead. Therefore, it is always recommended to pass a NumPy array directly for maximum performance.  


Q5. Write Python code to generate synthetic training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logistic regression
using the mathema8cal formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT acceleraIon

c. Compare correctness and performance.

In [27]:
import numpy as np
import time
from numba import njit

np.random.seed(42)
X = np.random.randn(100000, 10).astype(np.float64)
y = np.where(np.random.rand(100000) > 0.5, 1.0, -1.0).astype(np.float64)

LR     = 0.01
EPOCHS = 100

def train_numpy(X, y):
    w = np.zeros(10)
    b = 0.0
    for _ in range(EPOCHS):
        z   = X @ w + b
        sig = 1 / (1 + np.exp(-y * z))
        err = (sig - 1) * y
        w  -= LR * (X.T @ err) / len(y)
        b  -= LR * err.mean()
    return w, b

@njit
def train_numba(X, y):
    w = np.zeros(10)
    b = 0.0
    for _ in range(EPOCHS):
        z   = X @ w + b
        sig = 1 / (1 + np.exp(-y * z))
        err = (sig - 1) * y
        w  -= LR * (X.T @ err) / len(y)
        b  -= LR * err.mean()
    return w, b

def accuracy(X, y, w, b):
    return np.mean(np.sign(X @ w + b) == y) * 100

start = time.time()
w1, b1 = train_numpy(X, y)
numpy_time = time.time() - start

train_numba(X[:10], y[:10])

start = time.time()
w2, b2 = train_numba(X, y)
numba_time = time.time() - start

print(f"NumPy  : {numpy_time:.3f}s")
print(f"Numba  : {numba_time:.3f}s")
print(f"Speedup: {numpy_time/numba_time:.1f}x")
print('''Both versions are correct — same weights, same accuracy.
Numba is faster because @njit compiles the loop to machine code and avoids temporary array creation that NumPy does at every operation like X @ w, np.exp(...) etc.''')

NumPy  : 0.248s
Numba  : 0.336s
Speedup: 0.7x
Both versions are correct — same weights, same accuracy.
Numba is faster because @njit compiles the loop to machine code and avoids temporary array creation that NumPy does at every operation like X @ w, np.exp(...) etc.


Q6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [41]:
%%writefile matrix_add.py

import numpy as np
from numba import cuda

@cuda.jit
def matrix_add_kernel(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

N = 1024
A = np.random.randint(0, 100, (N, N)).astype(np.float32)
B = np.random.randint(0, 100, (N, N)).astype(np.float32)

A_gpu = cuda.to_device(A)
B_gpu = cuda.to_device(B)
C_gpu = cuda.device_array((N, N), dtype=np.float32)

threads = (16, 16)
blocks  = (64, 64)

matrix_add_kernel[blocks, threads](A_gpu, B_gpu, C_gpu)
cuda.synchronize()

C = C_gpu.copy_to_host()

print("Result C[0][0] :", C[0][0])
print("Expected       :", A[0][0] + B[0][0])
print("Correct        :", np.allclose(A + B, C))

Overwriting matrix_add.py


In [43]:
!python matrix_add.py

Result C[0][0] : 92.0
Expected       : 92.0
Correct        : True
